In [40]:
# Basic imports for API
import requests
import json
import pandas as pd

In [41]:
# Step 1: Authentication - Get the token
def get_acaps_token(username, password):
    url = "https://api.acaps.org/api/v1/token-auth/"
    payload = {
        'username': username,
        'password': password
    }
    
    response = requests.post(url, data=payload)
    
    if response.status_code == 200:
        # Extract the token from the response
        token = response.json().get('token')
        print(f"Token generated: {token}")
        return token
    elif response.status_code == 404:
        print("Error 404: URL not found. Check the URL.")
    elif response.status_code == 401:
        print("Error 401: Unauthorized. Check your credentials.")
    else:
        print(f"Error {response.status_code}: Unable to authenticate.")
        print(response.text)

In [42]:
# Step 2: Make a request using the token and handle pagination and save the data set in pandas data frame
def fetch_acaps_data(token, endpoint="countries", params=None):
    base_url = f"https://api.acaps.org/api/v1/{endpoint}/"
    headers = {
        'Authorization': f'Token {token}'
    }

    all_results = []
    while base_url:
        response = requests.get(base_url, headers=headers, params=params)
        
        if response.status_code == 200:
            data = response.json()
            # Append the results from the current page
            all_results.extend(data.get('results', []))
            # Get the URL for the next page
            base_url = data.get('next')
        else:
            print(f"Error: Failed to fetch data. Status code {response.status_code}")
            break
     # Convert the results to a DataFrame
    if all_results:
        df = pd.DataFrame(all_results)
        return df
    else:
        print("No data found")
        return None

In [43]:
# Step 3: Define your credentials
username = ""  # Replace with your actual username
password = ""    # Replace with your actual password

In [ ]:
# Step 4: Authenticate and get the token
token = get_acaps_token(username, password)

In [ ]:
# Step 4: Test API Access with token
def test_api_connection(token):
    url = "https://api.acaps.org/api/v1/countries/"  # Check the correct endpoint
    headers = {
        'Authorization': f'Token {token}'
    }
    
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        print("Successfully connected to the API.")
        print(response.json())
    elif response.status_code == 404:
        print("Error 404: URL not found. Check the endpoint.")
    elif response.status_code == 401:
        print("Error 401: Unauthorized. Token might be invalid.")
    else:
        print(f"Error {response.status_code}: Unable to access the API.")
        print(response.text)

if token:
    test_api_connection(token)


In [ ]:
if token:
    # Step 5: Fetch datasets (you can add filters and ordering in params)
    
    datasets = fetch_acaps_data(token, endpoint='yemen/core-dataset/economy') # Replace the endpoints with the dataset you want to access
    # Note: the full list of data sets are found on https://api.acaps.org/api/v1/ 

    # Step 6: Process and display the fetched data
    if datasets is not None:
        print(f"Fetched {datasets.shape[0]} records")

        # Optional: Save DataFrame to a CSV file if you need to
        datasets.to_csv('acaps_data.csv', index=False)
    else:
        print("No data found")

In [ ]:
datasets = datasets.dropna()
datasets.count()

In [ ]:
datasets.head(10)